# Check MEI and write an editorial report

**Workflow 1 — handle MEI files.** This notebook runs CAMAT's editorial checks on one or more MEI files and writes a CSV report. The MEI itself is not changed.

Use it on a single page, several pages, or a combined `*_full.mei`. The example is one facsimile-linked Buxtehude page in this repository.

Writes stay off until you set `RUN_CHECKS = True`. The report lands under `converted_mei/check_tutorial/` (gitignored).

**What you do**

1. Point `MEI_INPUTS` at the MEI file(s) to check.
2. Review the plan (nothing is written yet).
3. Set `RUN_CHECKS = True` and run the check cell.
4. Open the CSV in a spreadsheet (or read the preview below). Correct the MEI in [mei-friend](https://mei-friend.mdw.ac.at/), then re-run.

Each report row is one finding, with a severity of `error`, `warning`, or `info`. The suite covers identifiers and internal references, leftover import attributes, publication-profile rules, figured-bass anchors, page-break facsimile links, the packaged MEI 5.1 schema (`xmllint` required), and Verovio warnings.

Companion guide: [Handling MEI files](../docs/guides/edition-building.md). To join pages first, use [`mei_combine_pages.ipynb`](mei_combine_pages.ipynb). For extra flags (toggle individual checks, optional `<annot>` export, combine-then-check), use [`mei_consistency_checks.ipynb`](mei_consistency_checks.ipynb).

In [ ]:
# You can leave this cell unchanged.
try:
    from camat import (
        display_path,
        find_camat_root,
        report_summary,
        resolve_mei_inputs,
        resolve_repo_path,
        run_editorial_checks,
    )
except ModuleNotFoundError:
    import setup_camat
    from camat import (
        display_path,
        find_camat_root,
        report_summary,
        resolve_mei_inputs,
        resolve_repo_path,
        run_editorial_checks,
    )

## 1. Point at the MEI to check

`MEI_INPUTS` can be one `.mei` file, a folder of `*.mei` files, or a list of files. Repository-relative and absolute paths both work.

Review the plan, then set `RUN_CHECKS = True`.

In [ ]:
ROOT = find_camat_root()

# One file, a folder, or a list of .mei paths.
MEI_INPUTS = [
    "test_corpus/buxtehude_pages/bsb00023199_00126_facs_zones.mei",
]
# Combined score from mei_combine_pages.ipynb:
# MEI_INPUTS = ["converted_mei/combine_tutorial/buxtehude_pages_full.mei"]
# Several pages, or a folder:
# MEI_INPUTS = ["test_corpus/buxtehude_pages"]

# Where to write the CSV report (gitignored).
TARGET_DIR = "converted_mei/check_tutorial"

# Writes stay off until this is True.
RUN_CHECKS = False

## 2. Review the plan

This cell only lists the files that would be checked. It does not write a report or change the MEI.

In [ ]:
target_dir = resolve_repo_path(TARGET_DIR, repo_root=ROOT)
mei_files = resolve_mei_inputs(MEI_INPUTS, ROOT)
if not mei_files:
    raise FileNotFoundError("No .mei files found in MEI_INPUTS")

report_csv = (
    target_dir / f"{mei_files[0].stem}_consistency_report.csv"
    if len(mei_files) == 1
    else target_dir / "editorial_consistency_report.csv"
)

print(f"Files:      {len(mei_files)}")
for path in mei_files:
    print(f"  {display_path(path, repo_root=ROOT)}")
print(f"Report:     {display_path(report_csv, repo_root=ROOT)}")
print(f"Run enabled: {RUN_CHECKS}")

## 3. Run checks and write the report

With `RUN_CHECKS = True`, this cell writes a CSV under `TARGET_DIR` and prints a short summary. The listed MEI files are not rewritten.

In [ ]:
if not RUN_CHECKS:
    print("Skipped. Review the plan, then set RUN_CHECKS = True.")
    report_df = None
else:
    target_dir.mkdir(parents=True, exist_ok=True)
    report_df = run_editorial_checks(
        mei_files,
        root=ROOT,
        csv_out=report_csv,
    )
    print()
    print(report_summary(report_df).to_string(index=False))
    print()
    print(f"Report: {display_path(report_csv, repo_root=ROOT)}")
    print("Open that CSV in a spreadsheet. The MEI file was not changed.")

## 4. Look at the findings

Filter and sort the CSV in a spreadsheet while you correct the MEI. `error` rows need a fix; `warning` is worth a look; `info` is usually documentary. After you save the MEI, re-run this notebook until the rows you care about are gone.

The schema pass needs `xmllint`. If that check raises, install it and run again.

In [ ]:
if report_df is None:
    print("No report yet. Set RUN_CHECKS = True and run the cell above.")
elif report_df.empty:
    print("No findings. The report CSV was still written.")
else:
    preview_cols = [
        column
        for column in ("severity", "category", "check", "measure_n", "xml_id", "message")
        if column in report_df.columns
    ]
    print(f"{len(report_df)} finding(s). First rows (open the CSV for every column):")
    print(report_df[preview_cols].head(20).to_string(index=False))

## 5. What was produced?

| File | Meaning |
| --- | --- |
| `{stem}_consistency_report.csv` | findings for one input file |
| `editorial_consistency_report.csv` | findings when several files were checked together |

**Next steps**

1. Open the CSV, then correct the MEI in [mei-friend](https://mei-friend.mdw.ac.at/).
2. Inspect notation and facsimile links in [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb).
3. Re-run this notebook until the report is acceptable.

To check a different file, change `MEI_INPUTS`, set `RUN_CHECKS` back to `False` while you review the plan, then enable it again. To join pages before checking, use [`mei_combine_pages.ipynb`](mei_combine_pages.ipynb).